# Clasificador de Vocales (IY, IH, EH, AE, AH, AA, AO, UH, UW, ER)

Árbol de decisión entrenado con **F1, F2, F3**.

## 1. Carga del dataset

In [ ]:
import pandas as pd

df = pd.read_csv("dataset_solo_vocales.csv")

f1_candidates     = ["F1", "f1", "F1_Hz", "formant1"]
f2_candidates     = ["F2", "f2", "F2_Hz", "formant2"]
f3_candidates     = ["F3", "f3", "F3_Hz", "formant3"]
target_candidates = ["vocal", "Vocal", "target_vocal", "label_vocal"]

def find_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"No se encontró ninguna de: {candidates}. Disponibles: {list(df.columns)}")

f1_col  = find_column(df, f1_candidates)
f2_col  = find_column(df, f2_candidates)
f3_col  = find_column(df, f3_candidates)
target_col = find_column(df, target_candidates)

df_vocal = df[[f1_col, f2_col, f3_col, target_col]].dropna().copy()
X_vocal  = df_vocal[[f1_col, f2_col, f3_col]]
y_vocal  = df_vocal[target_col]

print(f"F1: {f1_col} | F2: {f2_col} | F3: {f3_col} | Target: {target_col}")
print(f"Shape: {df_vocal.shape}")
display(df_vocal.head())


## 2. Distribución de clases

Verifica que haya muestras suficientes para cada vocal.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

VOCALES_ORDER = ["IY","IH","EH","AE","AH","AA","AO","UH","UW","ER"]
counts = y_vocal.value_counts().reindex(VOCALES_ORDER, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(counts.index, counts.values,
              color=plt.cm.tab10.colors[:len(counts)],
              edgecolor="white", linewidth=0.8)
ax.bar_label(bars, fmt="%d", padding=4, fontsize=10)
ax.set_title("Distribución de muestras por vocal", fontsize=14, fontweight="bold")
ax.set_xlabel("Vocal (ARPABET)"); ax.set_ylabel("Muestras")
ax.set_ylim(0, counts.max() * 1.2)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("distribucion_vocales.png", dpi=150, bbox_inches="tight")
plt.show()
print(counts.to_string())


## 3. Espacio vocálico F1-F2

Visualizacion fonética estándar: F2 crece hacia la izquierda, F1 hacia abajo.

In [ ]:
VOCALES_ORDER = ["IY","IH","EH","AE","AH","AA","AO","UH","UW","ER"]
colores = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(9, 7))
for i, vocal in enumerate(VOCALES_ORDER):
    sub = df_vocal[y_vocal == vocal]
    if sub.empty:
        continue
    ax.scatter(sub[f2_col], sub[f1_col],
               color=colores[i], label=vocal,
               s=80, alpha=0.75, edgecolors="white", linewidths=0.4)
    ax.annotate(vocal, (sub[f2_col].mean(), sub[f1_col].mean()),
                fontsize=12, fontweight="bold", color=colores[i],
                ha="center", va="center",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.6, lw=0))

ax.invert_xaxis(); ax.invert_yaxis()
ax.set_xlabel("F2 (Hz)", fontsize=12); ax.set_ylabel("F1 (Hz)", fontsize=12)
ax.set_title("Espacio vocálico F1-F2", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", ncol=2, fontsize=9)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("espacio_vocalico.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Separación de datos y entrenamiento

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(
    X_vocal, y_vocal, test_size=0.2, random_state=42, stratify=y_vocal
)

tree_vocal = DecisionTreeClassifier(criterion="gini", max_depth=5, random_state=42)
tree_vocal.fit(X_train_v, y_train_v)

print(f"Train: {X_train_v.shape[0]} | Test: {X_test_v.shape[0]}")
print("Modelo entrenado correctamente.")


## 5. Evaluación básica

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_v = tree_vocal.predict(X_test_v)
acc_v = accuracy_score(y_test_v, y_pred_v)

print(f"Accuracy: {acc_v:.4f}\n")
print("Reporte de clasificación:")
print(classification_report(y_test_v, y_pred_v))


## 6. Matriz de confusión

Izquierda: conteos. Derecha: normalizada por fila (qué proporción de cada vocal se clasifica correctamente).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

labels_v = [v for v in VOCALES_ORDER if v in y_vocal.unique()]
cm_v = confusion_matrix(y_test_v, y_pred_v, labels=labels_v)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# — conteos —
disp = ConfusionMatrixDisplay(confusion_matrix=cm_v, display_labels=labels_v)
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Conteos absolutos", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Predicción"); axes[0].set_ylabel("Real")
for text in axes[0].texts:
    text.set_fontsize(10)

# — normalizada —
row_sums = cm_v.sum(axis=1, keepdims=True)
cm_norm = np.where(row_sums == 0, 0, cm_v.astype(float) / row_sums)
im = axes[1].imshow(cm_norm, interpolation="nearest", cmap="RdYlGn", vmin=0, vmax=1)
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
n = len(labels_v)
axes[1].set_xticks(range(n)); axes[1].set_xticklabels(labels_v, fontsize=10)
axes[1].set_yticks(range(n));  axes[1].set_yticklabels(labels_v, fontsize=10)
for i in range(n):
    for j in range(n):
        color = "white" if cm_norm[i, j] > 0.6 else "black"
        axes[1].text(j, i, f"{cm_norm[i,j]:.2f}",
                     ha="center", va="center", fontsize=9,
                     fontweight="bold", color=color)
axes[1].set_title("Normalizada por vocal (recall)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Predicción"); axes[1].set_ylabel("Real")

plt.suptitle("Clasificador de Vocales — Matriz de confusión",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("confusion_matrix_vocales.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardada como confusion_matrix_vocales.png")


## 7. Métricas por vocal (Precision, Recall, F1)

Identifica qué vocales son más difíciles de clasificar.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision_v, recall_v, f1_v, support_v = precision_recall_fscore_support(
    y_test_v, y_pred_v, labels=labels_v, zero_division=0
)

metricas_v = pd.DataFrame({
    "Vocal":     labels_v,
    "Precision": precision_v,
    "Recall":    recall_v,
    "F1-score":  f1_v,
    "Soporte":   support_v.astype(int)
}).set_index("Vocal")

display(metricas_v.round(3))

# Barras agrupadas
x = np.arange(len(labels_v))
width = 0.25
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - width, precision_v, width, label="Precision", color="#4C72B0", edgecolor="white")
ax.bar(x,         recall_v,    width, label="Recall",    color="#55A868", edgecolor="white")
ax.bar(x + width, f1_v,        width, label="F1-score",  color="#DD8452", edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(labels_v, fontsize=11)
ax.set_ylim(0, 1.2)
ax.axhline(0.8, color="gray", linestyle="--", linewidth=0.8, label="Umbral 0.80")
ax.set_ylabel("Score"); ax.set_title("Métricas por vocal", fontsize=14, fontweight="bold")
ax.legend(); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("metricas_por_vocal.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Errores más frecuentes

Pares de vocales que el modelo confunde con mayor frecuencia.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

errores = []
for i, real in enumerate(labels_v):
    for j, pred in enumerate(labels_v):
        if i != j and cm_v[i, j] > 0:
            errores.append({"Real": real, "Predicha": pred, "Confusiones": cm_v[i, j]})

df_err = pd.DataFrame(errores).sort_values("Confusiones", ascending=False).head(10)

if df_err.empty:
    print("¡Sin errores en el conjunto de prueba!")
else:
    df_err["Par"] = df_err["Real"] + " → " + df_err["Predicha"]
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.barh(df_err["Par"][::-1], df_err["Confusiones"][::-1],
                   color="#C44E52", edgecolor="white")
    ax.bar_label(bars, fmt="%d", padding=4, fontsize=10)
    ax.set_xlabel("Número de confusiones")
    ax.set_title("Top 10 errores más frecuentes\n(Real → Predicha)", fontsize=13, fontweight="bold")
    ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig("errores_frecuentes_vocales.png", dpi=150, bbox_inches="tight")
    plt.show()
    display(df_err[["Real","Predicha","Confusiones"]].reset_index(drop=True))


## 9. Importancia de features (F1, F2, F3)

In [ ]:
importancias_v = pd.Series(tree_vocal.feature_importances_,
                            index=X_vocal.columns).sort_values()

fig, ax = plt.subplots(figsize=(6, 3))
importancias_v.plot(kind="barh", ax=ax,
                    color=["#4C72B0","#55A868","#DD8452"][:len(importancias_v)],
                    edgecolor="white")
ax.set_title("Importancia de formantes (Gini)", fontsize=13, fontweight="bold")
ax.set_xlabel("Importancia relativa")
ax.spines[["top","right"]].set_visible(False)
for i, v in enumerate(importancias_v):
    ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=11)
plt.tight_layout()
plt.savefig("importancia_formantes.png", dpi=150, bbox_inches="tight")
plt.show()


## 10. Árbol de decisión y guardado del modelo

In [ ]:
from sklearn.tree import plot_tree
import joblib

plt.figure(figsize=(30, 15), dpi=300)
plot_tree(tree_vocal, feature_names=list(X_vocal.columns),
          class_names=[str(c) for c in tree_vocal.classes_],
          filled=True, rounded=True, fontsize=4)
plt.title("Árbol de decisión — clasificador de vocales", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("arbol_decision_vocales.png", dpi=200, bbox_inches="tight")
plt.show()

joblib.dump(tree_vocal, "decision_tree_vocales.pkl")
print("Modelo guardado como decision_tree_vocales.pkl")
